<a href="https://colab.research.google.com/github/avocado-planet/09-RAG/blob/main/sample_rag_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# シンプルなRAGアプリケーション実装例

このノートブックでは、LangChain を使って **RAG (Retrieval-Augmented Generation / 検索拡張生成)** の最小構成を実装します。

## このノートブックで学ぶこと
LangChain の GitHub リポジトリ上の Markdown ドキュメントを題材として、以下の一連の流れを実装します。

1. **Document Loader** — GitHub からドキュメントを取得
2. **Text Splitter** — ドキュメントを検索に適したサイズに分割
3. **Embeddings** — テキストを意味ベクトルに変換
4. **Vector Store (Chroma)** — ベクトルを保存し類似検索できるようにする
5. **Retriever** — クエリに関連するドキュメントを取得
6. **RAG Chain** — 取得した文脈を LLM に渡して回答を生成

## RAG の全体フロー

```
┌─────────────┐    ┌──────────┐    ┌────────────┐    ┌──────────┐
│  ドキュメント  │ → │ 分割 +   │ → │ ベクトル    │ → │ ベクトル   │
│  (Markdown) │    │ Embedding│    │ データベース │    │ 検索     │
└─────────────┘    └──────────┘    └────────────┘    └─────┬────┘
                                                            │
                                                            ↓
         ┌───────────┐          ┌──────────────┐      ┌──────────┐
         │   回答    │ ← LLM ← │ プロンプト    │ ← ← │ 関連文脈  │
         └───────────┘          │ (文脈+質問)  │      └──────────┘
                                └──────────────┘
```

> **前提**: Google Colab で実行。`OPENAI_API_KEY` を Colab のシークレットに登録済みであること。


## 00. 必要なライブラリのインストール

RAG パイプラインで使うライブラリを一括インストールします。

| ライブラリ | 用途 |
|---|---|
| `langchain-core` | LangChain の共通基盤 (Runnable, Prompt など) |
| `langchain-openai` | OpenAI の LLM / Embedding 連携 |
| `langchain-community` | `GitLoader` など周辺機能 |
| `langchain-text-splitters` | テキスト分割ユーティリティ |
| `langchain-chroma` | Chroma ベクトルストア連携 |
| `GitPython` | `GitLoader` が内部で使用 |


In [1]:
!pip install -q \
    langchain-core \
    langchain-openai \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    GitPython


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB

## 01. OpenAI API キーの設定

Google Colab の「シークレット」機能から `OPENAI_API_KEY` を読み込み、環境変数にセットします。
未設定のまま実行するとエラーが出るので、事前に Colab の鍵アイコンから登録しておいてください。


In [3]:
import os
from google.colab import userdata

# Colab シークレットから API キーを取得
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY が設定されていません。"
        "Colab の左側の鍵アイコンからシークレットに登録してください。"
    )
os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY を環境変数にセットしました。")


OPENAI_API_KEY を環境変数にセットしました。


## 02. Document Loader — ドキュメントの取得

`GitLoader` を使って LangChain の公式リポジトリをクローンし、**Markdown ファイル (`.md`) のみ** を読み込みます。

### ポイント
- `file_filter` で読み込むファイルを絞り込むことで、不要なファイルを除外できる
- 読み込んだドキュメントは `Document` オブジェクトのリストとして返される
- 各 `Document` は `page_content` (本文) と `metadata` (ソース情報) を持つ


In [4]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    """Markdown ファイル (.md) のみを読み込み対象とするフィルタ"""
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",  # クローン元リポジトリ
    repo_path="./langchain",                                  # クローン先のローカルパス
    branch="master",                                          # 対象ブランチ
    file_filter=file_filter,                                  # .md のみを通す
)

raw_docs = loader.load()
print(f"読み込んだドキュメント数: {len(raw_docs)}")
print(f"最初のドキュメントのメタデータ: {raw_docs[0].metadata}")


読み込んだドキュメント数: 28
最初のドキュメントのメタデータ: {'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}


## 03. Text Splitter — ドキュメントの分割

長いドキュメントをそのままベクトル化すると、以下の問題があります。

- **検索精度の低下**: 1 つのドキュメントに多様な話題が含まれ、クエリとの類似度がぼやける
- **コンテキスト長の制約**: LLM に渡せるトークン数に上限がある

そのため、ドキュメントを適切なサイズの **チャンク (chunk)** に分割します。

### `RecursiveCharacterTextSplitter` を使う理由

元のコードでは `CharacterTextSplitter` を使っていましたが、ここでは `RecursiveCharacterTextSplitter` に変更しています。

| | `CharacterTextSplitter` | `RecursiveCharacterTextSplitter` ⭐ |
|---|---|---|
| 分割基準 | 単一の区切り文字 | 段落 → 行 → 文 → 単語の順で再帰的に試行 |
| 文脈保持 | 弱い (文の途中で切れることも) | 強い (意味の境界で分割しやすい) |
| 推奨度 | 単純用途向け | **RAG の標準的な選択** |

### `chunk_size` と `chunk_overlap` の指針
- `chunk_size`: 500〜2000 文字が一般的。今回は **1000**
- `chunk_overlap`: `chunk_size` の 10〜20%。今回は **200** (チャンク境界での情報分断を緩和)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # 1 チャンクの最大文字数
    chunk_overlap=200,      # チャンク間で重複させる文字数 (文脈の連続性を保つ)
    separators=["\n\n", "\n", "。", ".", " ", ""],  # 試行する区切り文字 (優先順)
)

docs = text_splitter.split_documents(raw_docs)
print(f"分割後のチャンク数: {len(docs)}")
print(f"\n--- 最初のチャンクの例 ---")
print(f"メタデータ: {docs[0].metadata}")
print(f"本文 (先頭200文字):\n{docs[0].page_content[:200]}...")


分割後のチャンク数: 109

--- 最初のチャンクの例 ---
メタデータ: {'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}
本文 (先頭200文字):
# Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context
...


## 04. Embeddings — テキストのベクトル化

**Embedding (埋め込み)** とは、テキストを固定長の数値ベクトルに変換する処理です。
意味が近いテキストは、ベクトル空間上でも近い位置に配置されます。

ここでは OpenAI の `text-embedding-3-small` を使います。
- 次元数: **1536**
- 軽量かつ高速で、多くの RAG 用途に十分な精度

まずは動作確認として、1 つのクエリをベクトル化してみます。


In [6]:
from langchain_openai import OpenAIEmbeddings

# Embedding モデルを初期化
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 動作確認: 1 つのクエリをベクトル化
query = "LangChain を利用する理由は何ですか?"
vector = embeddings.embed_query(query)

print(f"ベクトル次元数: {len(vector)}")
print(f"ベクトル先頭 5 要素: {vector[:5]}")
print(f"(全 {len(vector)} 次元の浮動小数点数)")


ベクトル次元数: 1536
ベクトル先頭 5 要素: [0.000766754150390625, 0.017242431640625, 0.029693603515625, 0.018798828125, 0.0615234375]
(全 1536 次元の浮動小数点数)


## 05. Vector Store (Chroma) — ベクトルデータベースの構築と検索

**Chroma** はローカルで動作する軽量なベクトルデータベースです。
`Chroma.from_documents(docs, embeddings)` を呼ぶと、

1. 各ドキュメント (チャンク) を `embeddings` でベクトル化
2. ベクトルと元テキスト・メタデータをセットで Chroma に保存

という処理が自動で行われます。

### Retriever とは
Vector Store を検索インターフェース (`Retriever`) に変換すると、
LangChain の他のコンポーネントとパイプラインで繋げられるようになります。

検索はデフォルトで **コサイン類似度** に基づく近傍検索が行われ、上位 4 件が返されます。


In [7]:
from langchain_chroma import Chroma

# ドキュメントをベクトル化して Chroma に保存
db = Chroma.from_documents(docs, embeddings)

# Retriever (検索インターフェース) に変換
retriever = db.as_retriever()

# 検索を実行
context_docs = retriever.invoke(query)

print(f"検索結果の件数: {len(context_docs)}\n")
for i, d in enumerate(context_docs, 1):
    print(f"--- 結果 {i} ---")
    print(f"ソース: {d.metadata.get('source')}")
    print(f"本文 (先頭150文字): {d.page_content[:150]}...\n")


検索結果の件数: 4

--- 結果 1 ---
ソース: README.md
本文 (先頭150文字): ## Why use LangChain?

LangChain helps developers build applications powered by LLMs through a standard interface for models, embeddings, vector store...

--- 結果 2 ---
ソース: README.md
本文 (先頭150文字): While the LangChain framework can be used standalone, it also integrates seamlessly with any LangChain product, giving developers a full suite of tool...

--- 結果 3 ---
ソース: libs/langchain_v1/README.md
本文 (先頭150文字): # 🦜️🔗 LangChain

[![PyPI - Version](https://img.shields.io/pypi/v/langchain?label=%20)](https://pypi.org/project/langchain/#history)
[![PyPI - License...

--- 結果 4 ---
ソース: README.md
本文 (先頭150文字): <br>

LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperable components and third-party ...



## 06. RAG Chain — 検索結果を文脈として LLM に渡す

いよいよ RAG の本体です。LangChain の **LCEL (LangChain Expression Language)** の
`|` (パイプ) 演算子で、以下のパイプラインを構築します。

```
入力クエリ
   ↓
① { "context": retriever, "question": RunnablePassthrough() }
      - retriever がクエリを受け取り、関連ドキュメントを返す
      - RunnablePassthrough はクエリをそのまま通す
   ↓
② prompt テンプレート
      - {context} と {question} を埋め込んで完成したプロンプトを作る
   ↓
③ ChatOpenAI (gpt-4o-mini)
      - プロンプトから回答を生成 (AIMessage)
   ↓
④ StrOutputParser
      - AIMessage を文字列に変換
   ↓
出力
```

### なぜ `RunnablePassthrough()` が必要?
辞書の値はチェーン実行時に `chain.invoke(query)` の `query` を受け取る **Runnable** である必要があります。
- `retriever` は `Runnable` なので OK
- 生の文字列変数を書くとチェーン定義時に固定されてしまい使い回せない
- `RunnablePassthrough()` は「入力をそのまま次に渡す Runnable」なので、`question` にクエリをそのまま流せる


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# プロンプトテンプレート
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に解答してください。

文脈"""
{context}
"""

質問: {question}
''')

# LLM (temperature=0 で出力を決定的に)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG チェーンの構築
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 実行
output = chain.invoke(query)
print(output)


LangChainを利用する理由は、開発者がLLM（大規模言語モデル）を活用したアプリケーションを構築するための標準インターフェースを提供し、モデル、埋め込み、ベクトルストアなどの要素を統合することで、AIアプリケーションの開発を簡素化するからです。また、LangChainは独立して使用できるだけでなく、他のLangChain製品とシームレスに統合できるため、開発者に包括的なツールセットを提供します。これにより、複雑なタスクを処理するエージェントの構築や、アプリケーションのデバッグ、デプロイメントが容易になります。


## 07. (改善) 引用元を表示する

RAG の大きな利点の 1 つは **回答の根拠を追跡できること** です。
しかし、上のシンプルなチェーンでは検索結果 (context) が回答生成後に失われてしまいます。

ここでは `RunnableParallel` を使って **「回答と参照ドキュメントの両方を返すチェーン」** に拡張します。


In [9]:
from langchain_core.runnables import RunnableParallel

# 回答生成部分 (先ほどの chain と同じ構造)
answer_chain = prompt | model | StrOutputParser()

# 「context と question を準備する」部分
setup = {"context": retriever, "question": RunnablePassthrough()}

# 回答と参照ドキュメントを両方返すチェーン
rag_chain_with_sources = (
    RunnableParallel(
        {"context": retriever, "question": RunnablePassthrough()}
    )
    .assign(answer=answer_chain)
)

result = rag_chain_with_sources.invoke(query)

print("=" * 50)
print("回答:")
print("=" * 50)
print(result["answer"])

print("\n" + "=" * 50)
print("参照したドキュメント:")
print("=" * 50)
for i, d in enumerate(result["context"], 1):
    print(f"\n[{i}] {d.metadata.get('source')}")
    print(f"    {d.page_content[:120]}...")


回答:
LangChainを利用する理由は、開発者が大規模言語モデル（LLM）を活用したアプリケーションを構築するための標準インターフェースを提供することです。これにより、モデル、埋め込み、ベクトルストアなどの要素を簡単に組み合わせて、AIアプリケーションの開発を簡素化できます。また、LangChainは他のLangChain製品とシームレスに統合できるため、開発者はLLMアプリケーションを構築するための包括的なツールセットを利用できます。さらに、LangSmithを使用することで、アプリケーションの開発、テスト、監視を迅速に行うことができます。

参照したドキュメント:

[1] README.md
    ## Why use LangChain?

LangChain helps developers build applications powered by LLMs through a standard interface for mo...

[2] README.md
    While the LangChain framework can be used standalone, it also integrates seamlessly with any LangChain product, giving d...

[3] libs/langchain_v1/README.md
    # 🦜️🔗 LangChain

[![PyPI - Version](https://img.shields.io/pypi/v/langchain?label=%20)](https://pypi.org/project/langcha...

[4] README.md
    <br>

LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperab...


## 08. まとめ

このノートブックでは、以下の 6 ステップで RAG を実装しました。

| ステップ | コンポーネント | 役割 |
|---|---|---|
| 1 | `GitLoader` | GitHub からドキュメントを取得 |
| 2 | `RecursiveCharacterTextSplitter` | 検索に適したサイズに分割 |
| 3 | `OpenAIEmbeddings` | テキストをベクトルに変換 |
| 4 | `Chroma` | ベクトルを保存・検索 |
| 5 | `as_retriever()` | クエリに関連するチャンクを取得 |
| 6 | LCEL チェーン | 文脈を LLM に渡して回答生成 |

### 次に試すと良いこと
- **チャンクサイズの調整**: `chunk_size` / `chunk_overlap` を変えて検索品質を比較
- **MMR 検索**: `db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20})` で検索結果の多様性を上げる
- **リランキング**: Cohere Rerank などで検索結果を再並び替え
- **永続化**: `Chroma(persist_directory="./chroma_db", ...)` でディスクに保存して再利用
- **評価**: LangSmith や Ragas で RAG の品質を定量評価

詳細は付属の `RAG_explanation.md` を参照してください。
